In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel, VarianceThreshold, SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from dotenv import load_dotenv
import joblib
from gene_thesaurus import GeneThesaurus


In [3]:
load_dotenv()

X_train_path = os.getenv("X_TRAIN_CSV_PATH")
y_train_path = os.getenv("Y_TRAIN_TSV_PATH")
X_test_path = os.getenv("X_TEST_CSV_PATH")
y_test_path = os.getenv("Y_TEST_TSV_PATH")
output_dir = os.getenv("OUTPUT_DIR")
random_seed = int(os.getenv("RANDOM_SEED"))
n_jobs = int(os.getenv("N_JOBS", -1))

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"X_test_path: {X_test_path}")
print(f"y_test_path: {y_test_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")

# Hardcoded variables
VARIANCE_THRESHOLD=0.01  # REMOVE NEAR-CONSTANT GENES
FEATURE_SELECTION_PERCENTILE=20  # KEEP TOP n% OF FEATURES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/transpose/transposed.csv
y_train_path: /home/mariya/Data/pallas/nf-core-gexml-output-train/class/subtypes.tsv
X_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/transpose/transposed.csv
y_test_path: /home/mariya/Data/pallas/nf-core-gexml-output-cal/class/subtypes.tsv
output_dir: /home/mariya/Data/pallas/training
random_seed: 42
n_jobs: 10


In [4]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
id,,,,,,,,,,,,,,,,,,,,,
16-002,-0.757569,-1.272276,4.140267,2.213714,4.160025,3.299255,0.481260,2.090945,3.934939,6.488050,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
16-003,-0.830307,-1.272276,3.745861,4.048689,4.583248,2.407065,0.555469,1.831292,4.144439,6.299000,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
16-006,-0.993373,-1.272276,3.967398,3.778363,3.726068,3.512587,-0.702183,1.839737,3.618242,6.648740,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
16-011,3.489946,-1.272276,4.765434,4.518690,5.635658,4.079910,-0.347905,1.452577,4.881239,7.486040,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
16-014,-0.343193,-1.272276,4.205763,4.230049,3.425566,4.760768,0.268334,1.965802,4.980062,6.632752,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJPHALL020038,-1.046631,-1.272276,4.987958,2.800014,2.492139,4.681932,0.731582,2.698159,4.630361,5.537228,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJPHALL020046,-1.272276,-1.272276,3.528657,4.802694,4.553781,5.778022,5.903577,3.364666,4.690158,6.811363,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJPHALL020047,-0.535062,-1.272276,2.761021,4.368666,3.934511,5.124498,5.848494,3.577533,5.221458,7.235146,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276


In [5]:
y_train = pd.read_csv(y_train_path, sep="\t", index_col=0)
y_train

,class
sample,
SJAML031398_D1,"AML, KMT2A-r"
SJBALL032059_D1,"B-ALL, PAX5alt"
SJAML074187_D1,"AML, KMT2A-r"
SJBALL031613_D1,"B-ALL, PAX5 P80R"
SJBALL021023_D1,"B-ALL, high hyperdiploid"
...,...
SJHBBBN9RBJ,"B-ALL, high hyperdiploid"
SJHBBBM5GW9,"B-ALL, high hyperdiploid"
SJHBBBM4X52,"B-ALL, ETV6::RUNX1"


In [6]:
# align the indices
X_train = X_train.loc[y_train.index]
X_train

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
SJAML031398_D1,-1.067115,-1.272276,3.170445,2.839840,2.580800,7.919443,-0.412095,6.385257,3.933500,5.322568,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJBALL032059_D1,-0.627313,-0.753728,5.117654,2.917646,2.655815,3.946584,2.310893,2.322834,4.518807,7.007504,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJAML074187_D1,-0.580833,-1.272276,5.193442,4.232523,3.097974,5.390670,0.073023,6.498703,5.032813,5.786283,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJBALL031613_D1,-0.104178,0.508358,4.355553,3.051652,3.346683,5.406420,3.516816,3.749459,4.536801,6.042040,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJBALL021023_D1,0.260058,-1.272276,4.619448,2.592655,2.717725,6.284758,1.829920,3.979266,5.002776,5.476413,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBN9RBJ,2.168022,-1.272276,4.437911,2.822417,4.448756,1.284551,2.702390,2.377162,4.867666,6.213499,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJHBBBM5GW9,-0.861989,-1.272276,3.940327,4.476769,3.555881,7.297963,1.779578,3.339204,2.320633,6.345418,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276
SJHBBBM4X52,-0.967703,-1.272276,3.886541,4.456117,3.926058,7.432112,0.545522,3.870786,4.956120,6.189670,...,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276,-1.272276


In [7]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, sep="\t", index_col=0)
y_test

,class
sample,
GE8322,"B-ALL, ETV6::RUNX1"
ALLT-346,"B-ALL, KMT2A-r"
ALLT-338,"B-ALL, ETV6::RUNX1"
GE0310,"B-ALL, ETV6::RUNX1"
ALLT-371,"B-ALL, high hyperdiploid"
...,...
SJHBBBM5TXY,"B-ALL, ETV6::RUNX1"
SJHBBBM4JQZ,"B-ALL, ETV6::RUNX1"
SJHBBBN9RDM,"B-ALL, high hyperdiploid"


In [8]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,...,ENSG00000293642,ENSG00000293661,ENSG00000293662,ENSG00000293663,ENSG00000293685,ENSG00000293689,ENSG00000293691,ENSG00000300510,ENSG00000310517,ENSG00000310525
sample,,,,,,,,,,,,,,,,,,,,,
GE8322,-1.360879,-1.360879,1.876515,2.396068,2.826868,3.928644,-1.360879,3.198651,5.664029,7.174585,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
ALLT-346,-0.205741,-1.360879,3.844458,4.784285,4.329843,8.751081,0.017732,4.334528,5.437355,5.938224,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
ALLT-338,-0.615128,-1.360879,3.703454,4.543369,4.504947,4.587304,-1.146148,1.223584,4.567810,6.690555,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
GE0310,-0.086102,-1.360879,2.884915,2.567989,2.898130,3.315679,-0.183107,3.343280,4.414819,6.837778,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
ALLT-371,-0.137235,-1.360879,3.317370,4.268388,4.569072,2.951060,-1.212998,2.282673,3.625221,6.169482,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SJHBBBM5TXY,-0.672414,-1.360879,4.488283,3.079167,4.827530,3.135535,5.556245,1.175050,3.174006,7.587610,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
SJHBBBM4JQZ,-1.200954,-1.360879,4.060331,4.448735,3.988273,4.620535,7.114197,1.078019,3.612270,7.031694,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879
SJHBBBN9RDM,0.192781,-1.360879,4.795517,2.537434,4.206296,2.715249,6.672203,1.360018,3.695412,6.263331,...,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879,-1.360879


In [9]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(4)

In [10]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array
y_test = y_test.iloc[:, 0].values
y_test

array(['B-ALL, ETV6::RUNX1', 'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, TCF3::PBX1', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, NUTM1-r', 'B-ALL, PAX5 P80R',
       'B-ALL, BCR::ABL1-like', 'B-ALL, ETV6::RUNX1', 'B-ALL, ZNF384-r',
       'B-ALL, ETV6::RUNX1', 'B-ALL, DUX4-r', 'B-ALL, BCR::ABL1',
       'B-ALL, MYC-r', 'B-ALL, hypodiploid', 'B-ALL, KMT2A-r',
       'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1', 'B-ALL, iAMP21',
       'B-ALL, IKZF1 N159Y', 'B-ALL, ETV6::RUNX1', 'B-ALL, BCR::ABL1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, BCR::ABL1',
       'B-ALL, BCR::ABL1-like', 'B-ALL, KMT2A-r', 'B-ALL, PAX5alt',
       'B-ALL, PAX5alt', 'B-ALL, KMT2A-r', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, KMT2A-r', 'B-AL

In [11]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS

# ----------------------
# Pipeline: Variance filter -> RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("variance_filter", VarianceThreshold(threshold=VARIANCE_THRESHOLD)), # remove near-constant genes
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs))
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': [None, 'balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=-n_jobs,
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, os.path.join(output_dir, "best_hyperparameters.pkl"))

# Save features
# First get features that passed the variance threshold
variance_mask = best_pipeline.named_steps['variance_filter'].get_support()
remaining_features = X_train.columns[variance_mask]

# Then get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = remaining_features[selected_mask]

joblib.dump(selected_features, os.path.join(output_dir, "selected_features.pkl"))

# Save the whole pipeline
joblib.dump(best_pipeline, os.path.join(output_dir, "best_pipeline.pkl"))

Fitting 4 folds for each of 30 candidates, totalling 120 fits


Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 5, 'rf__max_features': 'sqrt', 'rf__max_depth': None, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.9179922835873834


['/home/mariya/Data/pallas/training/best_pipeline.pkl']

In [12]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

# Read in "best_hyperparameters.pkl"
with open(os.path.join(output_dir, "best_hyperparameters.pkl"), "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=-n_jobs,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': None, 'class_weight': 'balanced'}


[Parallel(n_jobs=-10)]: Using backend LokyBackend with 5 concurrent workers.
[Parallel(n_jobs=-10)]: Done  40 out of  40 | elapsed:  5.7min finished


Cross-validation F1-weighted: 0.918 ± 0.011


['/home/mariya/Data/pallas/training/final_model.pkl']

In [13]:
# ----------------------
# Evaluate final model on test set
# ----------------------

# Use selected features for test set
y_pred_final = final_rf.predict(X_test_selected)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test_selected.shape}")

print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))


Number of selected features: 3934
Shape of X_test_selected: (486, 3934)

Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     1.000     1.000         1
     AML, CBFA2T3::GLIS2      1.000     1.000     1.000         1
        AML, CBFB::MYH11      1.000     1.000     1.000        10
           AML, CEBPAalt      1.000     0.667     0.800         3
            AML, KMT2A-r      1.000     0.938     0.968        16
            AML, NPM1alt      0.800     1.000     0.889         4
            AML, NUP98-r      1.000     1.000     1.000         3
     AML, RUNX1::RUNX1T1      1.000     1.000     1.000        14
        B-ALL, BCR::ABL1      0.889     0.762     0.821        21
   B-ALL, BCR::ABL1-like      0.891     0.875     0.883        56
           B-ALL, DUX4-r      1.000     1.000     1.000        24
      B-ALL, ETV6::RUNX1      0.909     1.000     0.952        70
 B-ALL, ETV6

In [14]:

# ----------------------
# Extract top genes from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
         Gene  Importance
3278    NUTM1    0.005752
1904   MEGF10    0.004386
3836    APELA    0.003897
2218    AGAP1    0.003043
1845    MEIS1    0.002929
3099  LDLRAD3    0.002919
2175   TSPAN7    0.002608
1043     CNN3    0.002543
3328     PBX1    0.002535
3474    GREB1    0.002510
358   RUNX1T1    0.002501
2898    PTPRM    0.002352
3262   KCNJ12    0.002315
122     STAP1    0.002281
2266  IGF2BP1    0.002198
153     HDAC9    0.002128
3207     TTC3    0.002127
552     SALL4    0.002060
2468     EBF1    0.002035
3297   ANKS1B    0.002027


In [15]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 3 out of 3934

Bottom 20 genes by importance (final model):
              Gene  Importance
2094         PTPRR    0.000005
2528         PMPCA    0.000005
3246         CHEK2    0.000005
3702       DENND1C    0.000005
3771       TMEM233    0.000005
1956        MRPS28    0.000004
2813          NAT1    0.000004
2808           GAA    0.000004
1154      SLC25A16    0.000004
3286        SMIM10    0.000004
1196         MOCS1    0.000004
1746         TIGD7    0.000004
202         NCKAP1    0.000004
3933          SOD2    0.000004
3397          SBK2    0.000003
3346          PLK5    0.000002
1900          BTF3    0.000002
1242        CSN1S1    0.000000
3865  HOXA10-HOXA9    0.000000
3897       CT45A10    0.000000
